In [1]:
%matplotlib inline

# 	决策树实验

## 实验目的
掌握决策树学习算法。
## 实验要求
编程实现基于基尼指数进行划分选择的未剪枝决策树学习算法。


## 上机内容1

结合搜索引擎阅读并理解以下代码。

并回答：以下代码和课上所授决策树内容有什么区别？

**注意，请将以下代码中random_state替换为自己学号。**

In [ ]:
%reset -f
# 导入所需的库
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

# 加载鸢尾花数据集
iris = load_iris()
X = iris.data # 特征矩阵
y = iris.target # 类别向量
feature_names = iris.feature_names # 特征名称

# 划分训练集和测试集
random_state = 31
np.random.seed(random_state) # 设置随机种子，保证可复现性
indices = np.random.permutation(len(X)) # 生成随机索引
split = int(len(X) * 0.8) # 设置划分比例为80%
X_train = X[indices[:split]] # 训练集特征
y_train = y[indices[:split]] # 训练集类别
X_test = X[indices[split:]] # 测试集特征
y_test = y[indices[split:]] # 测试集类别

# 定义计算基尼指数的函数
def gini(y):
    """
    输入：类别向量y
    输出：基尼指数
    """
    # 获取类别标签以及对应的样本数量
    unique, counts = np.unique(y, return_counts=True)
    freqs = counts / len(y)
    # 计算每个类别的频率，并根据基尼指数的公式计算基尼指数
    gini = 1 - np.sum(freqs**2)
    return gini

# 定义根据特征和阈值划分数据集的函数
def split_dataset(X, y, feature_index, threshold):
    """
    输入：特征矩阵X，类别向量y，特征索引feature_index，阈值threshold
    输出：划分后的左右子集(X_left, y_left), (X_right, y_right)
    """    # 根据特征和阈值对数据进行二分划分
    left_indices = X[:, feature_index] <= threshold # 左子集的索引
    right_indices = X[:, feature_index] > threshold # 右子集的索引
    X_left = X[left_indices] # 左子集特征
    y_left = y[left_indices] # 左子集类别
    X_right = X[right_indices] # 右子集特征
    y_right = y[right_indices] # 右子集类别
    return (X_left, y_left), (X_right, y_right)

# 定义寻找最佳划分特征和阈值的函数
def best_split(X, y):
    """
    输入：特征矩阵X，类别向量y
    输出：最佳划分特征best_feature，最佳划分阈值best_threshold，最佳划分基尼指数best_gini
    """
    # 初始化最佳划分参数
    best_feature = None 
    best_threshold = None 
    best_gini = 1 # 最大可能的基尼指数为1
    
    n_features = X.shape[1] # 特征的数量
    
    for feature_index in range(n_features): # 遍历每个特征
        feature_values = X[:, feature_index] # 获取该特征的所有取值
        possible_thresholds = np.unique(feature_values) # 获取该特征的所有可能的阈值
        
        for threshold in possible_thresholds: # 遍历每个阈值
            # 根据该特征和阈值划分数据集为左右两个子集
            (X_left, y_left), (X_right, y_right) = split_dataset(X, y, feature_index, threshold)
            if len(y_left) == 0 or len(y_right) == 0: # 如果某个子集为空，则跳过该划分
                continue
            # 计算左右子集的权重和基尼指数
            weight_left = len(y_left) / len(y)
            weight_right = len(y_right) / len(y)
            gini_left = gini(y_left)
            gini_right = gini(y_right)
            # 计算该划分的加权基尼指数
            weighted_gini = weight_left * gini_left + weight_right * gini_right
            # 如果该划分的基尼指数小于当前最佳划分的基尼指数，则更新最佳划分参数
            if weighted_gini < best_gini:
                best_feature = feature_index
                best_threshold = threshold
                best_gini = weighted_gini
    
    return best_feature, best_threshold, best_gini


# 定义构建决策树的函数
def build_tree(X, y, max_depth=5, min_samples_split=2):
    """
    输入：特征矩阵X，类别向量y，最大深度max_depth，最小划分样本数min_samples_split
    输出：决策树，以字典的形式表示
    """
    # 创建一个空字典，用于存储决策树的信息
    tree = {}
    
    # 如果节点中的数据属于同一类别，则将节点标记为叶子节点，并返回其类别标签
    if len(np.unique(y)) == 1:
        tree["type"] = "leaf"
        tree["class"] = y[0]
        return tree
    
    # 如果节点的深度达到了最大深度，则将节点标记为叶子节点，并返回其数据中出现最多的类别标签
    if max_depth == 0:
        tree["type"] = "leaf"
        tree["class"] = np.bincount(y).argmax()
        return tree
    
    # 如果节点的数据量小于最小划分样本数，则将节点标记为叶子节点，并返回其数据中出现最多的类别标签
    if len(y) < min_samples_split:
        tree["type"] = "leaf"
        tree["class"] = np.bincount(y).argmax()
        return tree
    
    # 否则，寻找最佳划分特征和阈值，并将节点分为左右两个子节点
    best_feature, best_threshold, best_gini = best_split(X, y)
    
    # 如果没有找到合适的划分，则将节点标记为叶子节点，并返回其数据中出现最多的类别标签
    if best_feature is None or best_threshold is None:
        tree["type"] = "leaf"
        tree["class"] = np.bincount(y).argmax()
        return tree
    
    # 否则，根据最佳划分特征和阈值划分数据集为左右两个子集
    (X_left, y_left), (X_right, y_right) = split_dataset(X, y, best_feature, best_threshold)
    
    # 将节点标记为内部节点，并存储其划分特征和阈值
    tree["type"] = "internal"
    tree["feature"] = feature_names[best_feature]
    tree["threshold"] = best_threshold
    
    # 递归地对左右子节点进行同样的操作，减少最大深度
    tree["left"] = build_tree(X_left, y_left, max_depth-1, min_samples_split)
    tree["right"] = build_tree(X_right, y_right, max_depth-1, min_samples_split)
    
    return tree


# 定义根据决策树对新数据进行预测的函数
def predict(tree, x):
    """
    输入：决策树tree，单个样本x（一维数组）
    输出：预测结果y_pred（整数）
    """
    
    # 如果当前节点是叶子节点，则返回其类别标签作为预测结果
    if tree["type"] == "leaf":
        return tree["class"]
    
    # 否则，根据当前节点的划分特征和阈值将样本分配到左右子节点中
    feature_index = feature_names.index(tree["feature"]) # 获取划分特征的索引
    threshold = tree["threshold"] # 获取划分阈值
    
    if x[feature_index] <= threshold: # 如果样本的特征值小于等于阈值，则分配到左子节点
        return predict(tree["left"], x)
    else: # 否则，分配到右子节点
        return predict(tree["right"], x)

# 使用训练数据集构建决策树
tree = build_tree(X_train, y_train, max_depth=3, min_samples_split=5)

# 使用测试数据集进行预测，并计算准确率
y_pred = [predict(tree, x) for x in X_test]
accuracy = accuracy_score(y_test, y_pred)
print("The accuracy of the decision tree is:", accuracy)


The accuracy of the decision tree is: 0.9


## 上机内容2
基于以下代码框架和上述示例代码实现DecisionTreeClassifier（做适当修改），将其用于西瓜数据集2.0的分类，并计算准确率。

In [6]:
X.shape[1]

NameError: name 'X' is not defined

In [7]:
import numpy as np
from sklearn.metrics import accuracy_score
# ==================== 西瓜数据集 2.0 ====================
# 特征说明：
# 色泽(0:青绿,1:乌黑,2:浅白)，根蒂(0:蜷缩,1:稍蜷,2:硬挺)，
# 敲声(0:浊响,1:沉闷,2:清脆)，纹理(0:清晰,1:稍糊,2:模糊)，
# 脐部(0:凹陷,1:稍凹,2:平坦)，触感(0:硬滑,1:软粘)，
# 密度(连续)，含糖率(连续)
# 类别：好瓜(1) / 坏瓜(0)
def load_watermelon():
    # 原始数据（字符串形式，这里直接给出数值编码后的数据）
    data = np.array([
        [0,0,0,0,0,0,0.697,0.460,1],
        [1,0,1,0,0,0,0.774,0.376,1],
        [1,0,0,0,0,0,0.634,0.264,1],
        [0,0,1,0,0,0,0.608,0.318,1],
        [2,0,0,0,0,0,0.556,0.215,1],
        [0,1,0,0,1,1,0.403,0.237,1],
        [1,1,0,1,1,1,0.481,0.149,1],
        [1,1,0,0,1,0,0.437,0.211,1],
        [1,1,1,1,1,0,0.666,0.091,0],
        [0,2,2,0,2,1,0.243,0.267,0],
        [2,2,2,2,2,0,0.245,0.057,0],
        [2,0,0,2,2,1,0.343,0.099,0],
        [0,1,0,1,0,0,0.639,0.161,0],
        [2,1,1,1,0,0,0.657,0.198,0],
        [1,1,0,0,1,1,0.360,0.370,0],
        [2,0,0,2,2,0,0.593,0.042,0],
        [0,0,1,1,1,0,0.719,0.103,0]
    ])
    X = data[:, :-1]
    y = data[:, -1].astype(int)
    feature_names = ['色泽', '根蒂', '敲声', '纹理', '脐部', '触感', '密度', '含糖率']
    # 前6个特征是类别型，后2个是连续型
    categorical_features = list(range(6))
    return X, y, feature_names, categorical_features
# ==================== 决策树分类器 ====================
class DecisionTreeClassifier:
    def __init__(self, max_depth=5, min_samples_split=2, categorical_features=None):
        """
        参数:
            max_depth: 树的最大深度
            min_samples_split: 节点再划分所需的最小样本数
            categorical_features: 类别型特征的索引列表，其余特征视为连续型
        """
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.categorical_features = categorical_features if categorical_features is not None else []
        self.tree_ = None          # 训练后存储决策树
        self.feature_names_ = None  # 存储特征名称（可选）
    def _gini(self, y):
        """计算基尼指数"""
        _, counts = np.unique(y, return_counts=True)
        probs = counts / len(y)
        return 1 - np.sum(probs ** 2)
    def _split_dataset(self, X, y, feature_idx, threshold, is_categorical):
        """根据特征类型划分数据集"""
        if is_categorical:
            # 离散特征：左子集为 == threshold 的样本，右子集为 != threshold 的样本
            left_mask = X[:, feature_idx] == threshold
        else:
            # 连续特征：左子集为 <= threshold 的样本
            left_mask = X[:, feature_idx] <= threshold
        right_mask = ~left_mask
        return (X[left_mask], y[left_mask]), (X[right_mask], y[right_mask])
    def _best_split(self, X, y):
        """寻找最佳划分特征和阈值（或离散值）"""
        best_feature = None
        best_threshold = None
        best_gini = 1.0
        best_is_categorical = False
        n_features = X.shape[1]
        for feature_idx in range(n_features):
            is_categorical = feature_idx in self.categorical_features
            feature_values = X[:, feature_idx]
            if is_categorical:
                # 离散特征：候选划分点为所有不同的取值
                candidates = np.unique(feature_values)
                for value in candidates:
                    (X_left, y_left), (X_right, y_right) = self._split_dataset(
                        X, y, feature_idx, value, is_categorical=True
                    )
                    if len(y_left) == 0 or len(y_right) == 0:
                        continue
                    w_left = len(y_left) / len(y)
                    w_right = len(y_right) / len(y)
                    gini = w_left * self._gini(y_left) + w_right * self._gini(y_right)
                    if gini < best_gini:
                        best_gini = gini
                        best_feature = feature_idx
                        best_threshold = value
                        best_is_categorical = True
            else:
                # 连续特征：排序后取相邻值的中点作为候选阈值
                sorted_values = np.sort(np.unique(feature_values))
                # 如果只有一个值则无法划分
                if len(sorted_values) <= 1:
                    continue
                thresholds = (sorted_values[:-1] + sorted_values[1:]) / 2.0
                for threshold in thresholds:
                    (X_left, y_left), (X_right, y_right) = self._split_dataset(
                        X, y, feature_idx, threshold, is_categorical=False
                    )
                    if len(y_left) == 0 or len(y_right) == 0:
                        continue
                    w_left = len(y_left) / len(y)
                    w_right = len(y_right) / len(y)
                    gini = w_left * self._gini(y_left) + w_right * self._gini(y_right)
                    if gini < best_gini:
                        best_gini = gini
                        best_feature = feature_idx
                        best_threshold = threshold
                        best_is_categorical = False
        return best_feature, best_threshold, best_gini, best_is_categorical
    def _build_tree(self, X, y, depth):
        """递归构建决策树"""
        # 停止条件
        if len(np.unique(y)) == 1:
            return {"type": "leaf", "class": y[0]}
        if depth >= self.max_depth:
            return {"type": "leaf", "class": np.bincount(y).argmax()}
        if len(y) < self.min_samples_split:
            return {"type": "leaf", "class": np.bincount(y).argmax()}
        best_feature, best_threshold, best_gini, is_categorical = self._best_split(X, y)
        if best_feature is None:
            return {"type": "leaf", "class": np.bincount(y).argmax()}
        # 划分数据集
        (X_left, y_left), (X_right, y_right) = self._split_dataset(
            X, y, best_feature, best_threshold, is_categorical
        )
        # 构建内部节点
        node = {
            "type": "internal",
            "feature_idx": best_feature,
            "threshold": best_threshold,
            "is_categorical": is_categorical
        }
        # 递归构建左右子树
        node["left"] = self._build_tree(X_left, y_left, depth + 1)
        node["right"] = self._build_tree(X_right, y_right, depth + 1)
        return node
    def fit(self, X, y, feature_names=None):
        """
        训练决策树
        X: 特征矩阵 (numpy array)，离散特征需已编码为整数
        y: 标签向量
        feature_names: 特征名称列表（可选，仅用于可读性）
        """
        self.feature_names_ = feature_names
        self.tree_ = self._build_tree(X, y, depth=0)
    def _predict_one(self, x, node):
        """对单个样本递归预测"""
        if node["type"] == "leaf":
            return node["class"]
        feature_val = x[node["feature_idx"]]
        if node["is_categorical"]:
            if feature_val == node["threshold"]:
                return self._predict_one(x, node["left"])
            else:
                return self._predict_one(x, node["right"])
        else:
            if feature_val <= node["threshold"]:
                return self._predict_one(x, node["left"])
            else:
                return self._predict_one(x, node["right"])
    def predict(self, X):
        """对多个样本预测"""
        return [self._predict_one(x, self.tree_) for x in X]
# ==================== 在西瓜数据集上测试 ====================
if __name__ == "__main__":
    # 加载数据
    X, y, feature_names, cat_features = load_watermelon()
    # 创建分类器
    clf = DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=2,
        categorical_features=cat_features
    )
    # 训练（在此使用全部数据训练，演示训练集准确率）
    clf.fit(X, y, feature_names)
    # 预测
    y_pred = clf.predict(X)
    # 计算准确率
    acc = accuracy_score(y, y_pred)
    print(f"训练集准确率: {acc:.4f}")
    # 可选：打印树结构（简单显示）
    def print_tree(node, depth=0, prefix=""):
        if node["type"] == "leaf":
            print(" " * depth + f"{prefix} -> 类别: {node['class']}")
        else:
            feat_name = feature_names[node["feature_idx"]] if feature_names else f"X{node['feature_idx']}"
            if node["is_categorical"]:
                cond = f"{feat_name} == {node['threshold']}"
            else:
                cond = f"{feat_name} <= {node['threshold']:.3f}"
            print(" " * depth + f"{prefix}{cond}")
            print_tree(node["left"], depth + 2, "是 -> ")
            print_tree(node["right"], depth + 2, "否 -> ")
    print("\n生成的决策树结构:")
    print_tree(clf.tree_)

训练集准确率: 1.0000

生成的决策树结构:
纹理 == 0.0
  是 -> 密度 <= 0.382
    是 ->  -> 类别: 0
    否 ->  -> 类别: 1
  否 -> 色泽 == 1.0
    是 -> 敲声 == 0.0
      是 ->  -> 类别: 1
      否 ->  -> 类别: 0
    否 ->  -> 类别: 0
